# Corporate Sales Record Cleaner (Automated Data Sanitization Pipeline)
**GetIntern.in | Week 1 Milestone Project | Phase 1: Data Cleaning & EDA**

This notebook cleans raw e-commerce data (`transactions.csv` and `customers.csv`) using **Pandas** and **NumPy**.

**Objectives**
1. Ingestion & duplicate removal
2. Missing value imputation
3. Text standardization & export

## Setup
### Step 1: Import libraries
Pandas is used for the data work and NumPy for handling missing values.

In [1]:
import pandas as pd
import numpy as np

### Step 2: Load the raw data
Both raw files are loaded with `pd.read_csv()`. `phone_number` is read as text so leading zeros are not lost. A working copy (`dfs`) is made so the raw data stays untouched for the before/after comparison.

In [3]:
raw = {
    "transactions": pd.read_csv("transactions.csv"),
    "customers": pd.read_csv("customers.csv", dtype={"phone_number": str}),
}
dfs = {name: d.copy() for name, d in raw.items()}

for name, d in raw.items():
    print(name, d.shape)
    print(d.columns.tolist())
    display(d.head())

transactions (8199, 10)
['transaction_id', 'customer_id', 'transaction_date', 'product_id', 'quantity', 'unit_price', 'payment_method', 'discount_applied', 'transaction_status', 'review_text']


,transaction_id,customer_id,transaction_date,product_id,quantity,unit_price,payment_method,discount_applied,transaction_status,review_text
0,TX_0000003568,CUST0429,2024-12-29,WZ-802,30,448000,Credit Card,6.0,Completed,NaN
1,TX_0000001323,CUST0165,2025-09-20,LE-969,31,326000,E-wallet,7.0,Completed,NaN
2,TX_0000007976,CUST0969,2025-09-19,CR-562,22,326000,E-wallet,8.0,Completed,NaN
3,TX_0000002408,CUST0293,2025-04-23,BL-316,38,418000,Credit Card,42.0,Completed,NaN
4,TX_0000004435,CUST0537,2025-08-28,KL-203,3,15000,E-wallet,27.0,Completed,NaN


customers (1010, 9)
['customer_id', 'name', 'age', 'gender', 'state', 'signup_date', 'email', 'phone_number', 'subscribe']


,customer_id,name,age,gender,state,signup_date,email,phone_number,subscribe
0,CUST0001,Gandi,26.0,Male,Sulawesi Utara,2025-05-05,gandi_96@example.com,08422757160,No
1,CUST0002,Cecep,49.0,Male,Aceh,2025-06-02,cecep467@example.com,088171122035,No
2,CUST0003,Artawan,26.0,Male,Jambi,2025-02-11,wahyuninovi@example.net,08692932468,Yes
3,CUST0004,Mulya,61.0,Male,Nusa Tenggara Timur,2025-02-19,nnapitupulu@example.com,08515735210,Yes
4,CUST0005,Taufik,43.0,Male,Kepulauan Riau,2025-08-04,taufik_87@example.com,081034538051,Yes


### Step 3: Tidy column names
Column names are stripped, converted to lowercase, and spaces are replaced with underscores so they are consistent.

In [4]:
for name, d in dfs.items():
    d.columns = d.columns.str.strip().str.lower().str.replace(" ", "_")
    print(name, d.columns.tolist())

transactions ['transaction_id', 'customer_id', 'transaction_date', 'product_id', 'quantity', 'unit_price', 'payment_method', 'discount_applied', 'transaction_status', 'review_text']
customers ['customer_id', 'name', 'age', 'gender', 'state', 'signup_date', 'email', 'phone_number', 'subscribe']


## Objective 1: Ingestion & Duplicate Removal
### Step 4: Audit duplicate rows
Count fully identical rows in each file using `.duplicated().sum()`.

In [5]:
for name, d in dfs.items():
    print(name, "duplicate rows:", d.duplicated().sum())

transactions duplicate rows: 0
customers duplicate rows: 10


### Step 5: Remove duplicate rows
Redundant rows are removed with `.drop_duplicates()`. The customers file had 10 duplicates (1,010 to 1,000 rows). The transactions file had none.

In [6]:
for name, d in dfs.items():
    dfs[name] = d.drop_duplicates()
    print(name, "shape after dedup:", dfs[name].shape)

transactions shape after dedup: (8199, 10)
customers shape after dedup: (1000, 9)


### Step 5b: Check for repeated IDs
Besides identical rows, we check that `transaction_id` and `customer_id` are not repeated. Both counts are 0, so no extra records need removing.

In [7]:
print("Duplicate transaction_ids:", dfs["transactions"].duplicated(subset="transaction_id").sum())
print("Duplicate customer_ids:", dfs["customers"].duplicated(subset="customer_id").sum())

Duplicate transaction_ids: 0
Duplicate customer_ids: 0


## Objective 2: Missing Value Imputation
### Step 6: Convert placeholder text to real nulls
Values such as "unknown", "error", "n/a" and empty strings are converted to `NaN` so they are counted as missing.

In [8]:
placeholders = ["unknown", "error", "n/a", "na", "none", "null", "nan", "-", ""]
for name, d in dfs.items():
    for col in d.select_dtypes(include="object").columns:
        c = d[col].astype(str).str.strip().str.lower()
        d.loc[c.isin(placeholders), col] = np.nan

### Step 7: Define a helper for ID and date columns
ID, date and phone columns should not be filled or converted like normal numbers or text. This helper function identifies them so later steps can skip them.

In [9]:
def is_id_or_date(col):
    return ("date" in col or "phone" in col or col == "id"
            or col.endswith("_id") or col.startswith("id_"))

### Step 8: Convert date columns
`transaction_date` and `signup_date` are converted to datetime so they have the correct data type.

In [10]:
for name, d in dfs.items():
    for col in d.columns:
        if "date" in col:
            d[col] = pd.to_datetime(d[col], errors="coerce")
            print(name, "->", col, "converted to datetime")

transactions -> transaction_date converted to datetime
customers -> signup_date converted to datetime


### Step 9: Inspect null values
Missing values are counted per column with `df.isnull().sum()`:
- transactions: `discount_applied` (426), `review_text` (4,916)
- customers: `age` (56), `state` (53)

In [11]:
for name, d in dfs.items():
    print(name)
    print(d.isnull().sum(), "\n")

transactions
transaction_id           0
customer_id              0
transaction_date         0
product_id               0
quantity                 0
unit_price               0
payment_method           0
discount_applied       426
transaction_status       0
review_text           4916
dtype: int64 

customers
customer_id      0
name             0
age             56
gender           0
state           53
signup_date      0
email            0
phone_number     0
subscribe        0
dtype: int64 



### Step 10: Fill numerical columns with the median
`age` and `discount_applied` are filled with the column **median**, because the median is not affected by outliers or skewed values.

In [12]:
num_cols = {}
for name, d in dfs.items():
    num_cols[name] = list(d.select_dtypes(include="number").columns)
    for col in num_cols[name]:
        d[col] = d[col].fillna(d[col].median())
    print(name, "median-filled:", num_cols[name])

transactions median-filled: ['quantity', 'unit_price', 'discount_applied']
customers median-filled: ['age']


### Step 11: Fill categorical columns
- Most categorical columns (such as `state`) are filled with the **mode**, the most frequent value.
- `review_text` is filled with the placeholder **"no review"**. About 60% of reviews are missing, and filling them with the most common review would invent data.

In [13]:
cat_cols = {}
for name, d in dfs.items():
    cat_cols[name] = [c for c in d.select_dtypes(include="object").columns if not is_id_or_date(c)]
    for col in cat_cols[name]:
        if col == "review_text":
            d[col] = d[col].fillna("no review")      # placeholder, not mode
        else:
            d[col] = d[col].fillna(d[col].str.strip().str.lower().mode()[0])
    print(name, "filled:", cat_cols[name])

transactions filled: ['payment_method', 'transaction_status', 'review_text']
customers filled: ['name', 'gender', 'state', 'email', 'subscribe']


### Step 12: Fill missing dates
Any missing date is filled with the most frequent date. No dates were missing in this dataset, so no values changed.

In [14]:
for name, d in dfs.items():
    for col in d.columns:
        if "date" in col and d[col].isnull().any():
            d[col] = d[col].fillna(d[col].mode()[0])

### Step 13: Verify no null values remain
Both files should show 0 remaining nulls.

In [15]:
for name, d in dfs.items():
    print(name, "remaining nulls:", d.isnull().sum().sum())

transactions remaining nulls: 0
customers remaining nulls: 0


### Step 14: Fix the data type of `age`
After filling, `age` is converted from decimal (26.0) to whole numbers (26).

In [16]:
dfs["customers"]["age"] = dfs["customers"]["age"].astype(int)
print(dfs["customers"]["age"].dtype)

int64


## Objective 3: Text Standardization & Export
### Step 15: Standardize text
Text columns are cleaned with `.str.strip()` to remove extra whitespace and `.str.lower()` to make them lowercase, so the same value is always written the same way (for example `Credit Card` becomes `credit card`).

In [17]:
for name, d in dfs.items():
    for col in cat_cols[name]:
        d[col] = d[col].str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
        print(name, col, "->", d[col].unique()[:8])

transactions payment_method -> ['credit card' 'e-wallet' 'cash' 'bank transfer' 'debit card']
transactions transaction_status -> ['completed' 'pending' 'cancelled' 'refunded']
transactions review_text -> ['no review' 'very slow shipping' 'a bit expensive but worth the quality'
 'amazing product, exceeds expectations' 'product matches description'
 'honest seller, item as described'
 'item damaged upon arrival, disappointed' "won't buy here again"]
customers name -> ['gandi' 'cecep' 'artawan' 'mulya' 'taufik' 'genta' 'dimas' 'nilam']
customers gender -> ['male' 'female']
customers state -> ['sulawesi utara' 'aceh' 'jambi' 'nusa tenggara timur' 'kepulauan riau'
 'kalimantan timur' 'sumatera barat' 'jawa timur']
customers email -> ['gandi_96@example.com' 'cecep467@example.com' 'wahyuninovi@example.net'
 'nnapitupulu@example.com' 'taufik_87@example.com' 'puyainah@example.com'
 '224dimas@example.com' 'tariandriani@example.org']
customers subscribe -> ['no' 'yes']


### Step 16: Re-check for duplicates
Text cleaning can make near-duplicate rows identical, so duplicates are checked again. 0 extra duplicates were found.

In [18]:
for name, d in dfs.items():
    before = len(d)
    dfs[name] = d.drop_duplicates()
    print(name, "extra duplicates removed:", before - len(dfs[name]))

transactions extra duplicates removed: 0
customers extra duplicates removed: 0


### Step 17: Baseline summary statistics
`.describe()` gives summary statistics (count, mean, min, max and quartiles) for the cleaned data.

In [19]:
for name, d in dfs.items():
    print(name, "final shape:", d.shape)
    display(d.describe())

transactions final shape: (8199, 10)


,transaction_date,quantity,unit_price,discount_applied
count,8199,8199.000000,8.199000e+03,8199.000000
mean,2025-03-04 13:46:41.756311808,22.376265,2.942082e+05,25.022442
min,2023-10-17 00:00:00,1.000000,1.000000e+02,0.000000
25%,2024-12-03 00:00:00,16.000000,1.750000e+05,13.000000
50%,2025-02-14 00:00:00,22.000000,2.830000e+05,25.000000
75%,2025-06-29 00:00:00,30.000000,3.950000e+05,37.000000
max,2025-09-24 00:00:00,40.000000,2.000000e+06,50.000000
std,NaN,10.060358,1.874184e+05,14.229293


customers final shape: (1000, 9)


,age,signup_date
count,1000.000000,1000
mean,41.192000,2024-09-23 00:56:09.600000
min,18.000000,2023-09-24 00:00:00
25%,29.000000,2024-04-08 00:00:00
50%,42.000000,2024-09-21 00:00:00
75%,52.000000,2025-03-18 06:00:00
max,65.000000,2025-09-23 00:00:00
std,13.483061,NaN


### Step 18: Export the cleaned datasets
The sanitized files are saved as `cleaned_transactions.csv`, `cleaned_customers.csv` and `cleaned_sales.csv` (the final sales dataset).

In [20]:
dfs["transactions"].to_csv("cleaned_transactions.csv", index=False)
dfs["customers"].to_csv("cleaned_customers.csv", index=False)
dfs["transactions"].to_csv("cleaned_sales.csv", index=False)   # name used in the task email
print("Saved all cleaned files")

Saved all cleaned files


## Before vs After Comparison
This table compares each file before and after cleaning (rows, columns, null values and duplicates).

In [21]:
rows = []
for name in raw:
    rows.append({
        "File": name,
        "Raw rows": raw[name].shape[0],
        "Cleaned rows": dfs[name].shape[0],
        "Columns": dfs[name].shape[1],
        "Raw nulls": int(raw[name].isnull().sum().sum()),
        "Cleaned nulls": int(dfs[name].isnull().sum().sum()),
        "Raw duplicates": int(raw[name].duplicated().sum()),
    })
pd.DataFrame(rows)

,File,Raw rows,Cleaned rows,Columns,Raw nulls,Cleaned nulls,Raw duplicates
0,transactions,8199,8199,10,5342,0,0
1,customers,1010,1000,9,110,0,10


## Conclusion
- Removed **10 duplicate rows** from the customers data.
- Filled all **missing values** (0 nulls remain in both files):
  - Numerical columns: median
  - Categorical columns: mode, with a "no review" placeholder for `review_text`
- Standardized text (stripped whitespace, lowercase) and fixed data types.
- Exported the final sanitized dataset as `cleaned_sales.csv`.